# GPU 與 TPU 比較

在 Colab 中選擇 GPU 或 TPU 執行階段，分別執行此筆記本並下載結果。此試驗使用相同的合成資料與小型模型，測量訓練速度；它不是 BoolieLOT 的 gpt-oss-20b 訓練。兩次使用相同設定，記錄裝置與 Colab 執行階段類型。初始化與編譯時間另列，穩態速度不包含首次編譯。免費資源型號和可用時間會變動。

In [ ]:
import os, time, json, platform
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("TPU address available:", bool(os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME")))

In [ ]:
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
except ValueError:
    resolver = None
start = time.perf_counter()
if resolver is not None:
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    device_kind = "TPU"
elif tf.config.list_physical_devices("GPU"):
    strategy = tf.distribute.OneDeviceStrategy("/GPU:0")
    device_kind = "GPU"
else:
    raise RuntimeError("請在 Colab 執行階段設定選擇 GPU 或 TPU。")
print(device_kind, "replicas:", strategy.num_replicas_in_sync, "setup seconds:", time.perf_counter()-start)

## 相同工作量

每步使用固定全域批次及相同網路；TPU 多核心將批次分配至各核心。使用固定資料避免把下載速度算入結果；數值精度為 float32，結果不可推論其他模型的表現。

In [ ]:
BATCH, FEATURES, CLASSES = 256, 1024, 128
WARMUP, STEPS = 10, 100
tf.random.set_seed(1899)
with strategy.scope():
    model = tf.keras.Sequential([tf.keras.layers.Input((FEATURES,)), tf.keras.layers.Dense(1024, activation="relu"), tf.keras.layers.Dense(CLASSES)])
    optimizer = tf.keras.optimizers.Adam()
    model.compile(optimizer=optimizer, loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), steps_per_execution=1)
x = tf.random.stateless_normal([4096, FEATURES], seed=[1, 2])
y = tf.random.stateless_uniform([4096], seed=[3, 4], maxval=CLASSES, dtype=tf.int32)
ds = tf.data.Dataset.from_tensor_slices((x, y)).repeat().batch(BATCH, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
start = time.perf_counter()
model.fit(ds, epochs=1, steps_per_epoch=WARMUP, verbose=0)
warmup_seconds = time.perf_counter()-start
start = time.perf_counter()
history = model.fit(ds, epochs=1, steps_per_epoch=STEPS, verbose=0)
elapsed = time.perf_counter()-start
result = {"device":device_kind, "devices":[d.name for d in tf.config.list_logical_devices(device_kind)], "replicas":strategy.num_replicas_in_sync, "tensorflow":tf.__version__, "batch":BATCH, "warmup_steps":WARMUP, "measured_steps":STEPS, "warmup_seconds":warmup_seconds, "measured_seconds":elapsed, "examples_per_second":STEPS*BATCH/elapsed, "final_loss":float(history.history["loss"][-1])}
print(json.dumps(result, indent=2))
with open("benchmark-result.json", "w") as f: json.dump(result, f, indent=2)

In [ ]:
from google.colab import files
files.download("benchmark-result.json")

## GPU 與 TPU 並用

一般 Colab 執行階段選擇一種加速器；此筆記本不假設同時提供兩者。跨裝置協同需要特別支援的軟體、硬體及資料交換，不能把兩次速度直接相加。BoolieLOT 現行 Unsloth、CUDA 和 4-bit 訓練流程是 GPU 路徑，TPU 需要另行移植並驗證；此試驗不測試兩者同時訓練。比較時也應看費用、初始化、記憶體和模型品質。